# EXP-028 | 시드 앙상블 (Seed Ensemble)

EXP-020 파라미터(최고 점수 0.74217) 그대로, 시드만 바꿔서 학습 후 평균.
5개 시드 × 3모델 = 15개 모델 앙상블. 랜덤성으로 인한 분산을 줄여 안정적인 gain 확보.

| 모델 | 파라미터 출처 |
|---|---|
| LightGBM | EXP-019 Optuna |
| CatBoost | EXP-011 Optuna |
| XGBoost | EXP-012 Optuna |

| 시드 | [42, 0, 123, 2024, 777] |
|---|---|
| 기준선 | EXP-020 = 0.74068 (OOF) / 0.74217 (해커톤) |

In [8]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
import time
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                              precision_score, accuracy_score)
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
N_FOLDS  = 5
EXP_NO   = 28
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

SEEDS = [42, 0, 123, 2024, 777]

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')
print(f'시드: {SEEDS}  →  총 {len(SEEDS) * 3}개 모델')

train: (256351, 69)  /  test: (90067, 68)
시드: [42, 0, 123, 2024, 777]  →  총 15개 모델


## 1. 피처 준비 (EXP-020 동일)

In [9]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']     = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. 모델 파라미터 (EXP-020 동일)

In [10]:
LGB_PARAMS_BASE = dict(
    objective='binary', metric='auc', verbosity=-1,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS_BASE = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', thread_count=-1, verbose=False,
    early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS_BASE = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist',
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)
print('파라미터 설정 완료 (EXP-020 기준)')

파라미터 설정 완료 (EXP-020 기준)


## 3. 시드 앙상블 학습

In [11]:
oof_lgb  = np.zeros(len(X_train))
oof_cat  = np.zeros(len(X_train))
oof_xgb  = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

t_total = time.time()
for s_idx, seed in enumerate(SEEDS, 1):
    print(f'\n=== Seed {seed} ({s_idx}/{len(SEEDS)}) ===')
    lgb_p = {**LGB_PARAMS_BASE, 'seed': seed}
    cat_p = {**CAT_PARAMS_BASE, 'random_seed': seed}
    xgb_p = {**XGB_PARAMS_BASE, 'random_state': seed}

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_lgb_s = np.zeros(len(X_train))
    oof_cat_s = np.zeros(len(X_train))
    oof_xgb_s = np.zeros(len(X_train))
    test_lgb_s = np.zeros(len(X_test))
    test_cat_s = np.zeros(len(X_test))
    test_xgb_s = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr  = X_train.iloc[tr_idx]
        X_val = X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        # LGB
        ds_tr  = lgb.Dataset(X_tr, label=y_tr)
        ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
        m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                 lgb.log_evaluation(-1)])
        oof_lgb_s[val_idx]  = m.predict(X_val)
        test_lgb_s         += m.predict(X_test) / N_FOLDS

        # CAT
        m = CatBoostClassifier(**cat_p)
        m.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
        oof_cat_s[val_idx]  = m.predict_proba(X_val)[:, 1]
        test_cat_s         += m.predict_proba(X_test)[:, 1] / N_FOLDS

        # XGB
        m = xgb.XGBClassifier(**xgb_p)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb_s[val_idx]  = m.predict_proba(X_val)[:, 1]
        test_xgb_s         += m.predict_proba(X_test)[:, 1] / N_FOLDS

    auc_lgb_s = roc_auc_score(y_train, oof_lgb_s)
    auc_cat_s = roc_auc_score(y_train, oof_cat_s)
    auc_xgb_s = roc_auc_score(y_train, oof_xgb_s)
    rank_s = np.stack([rankdata(oof_lgb_s), rankdata(oof_cat_s), rankdata(oof_xgb_s)], axis=1).mean(axis=1)
    print(f'  LGB={auc_lgb_s:.5f}  CAT={auc_cat_s:.5f}  XGB={auc_xgb_s:.5f}  Rank={roc_auc_score(y_train, rank_s):.5f}')

    oof_lgb  += oof_lgb_s  / len(SEEDS)
    oof_cat  += oof_cat_s  / len(SEEDS)
    oof_xgb  += oof_xgb_s  / len(SEEDS)
    test_lgb += test_lgb_s / len(SEEDS)
    test_cat += test_cat_s / len(SEEDS)
    test_xgb += test_xgb_s / len(SEEDS)

print(f'\n총 소요: {(time.time()-t_total)/60:.1f}분')
auc_lgb = roc_auc_score(y_train, oof_lgb)
auc_cat = roc_auc_score(y_train, oof_cat)
auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'평균 OOF  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')


=== Seed 42 (1/5) ===
  LGB=0.74047  CAT=0.74015  XGB=0.74051  Rank=0.74068

=== Seed 0 (2/5) ===
  LGB=0.74014  CAT=0.74006  XGB=0.74025  Rank=0.74045

=== Seed 123 (3/5) ===
  LGB=0.74030  CAT=0.73997  XGB=0.74027  Rank=0.74048

=== Seed 2024 (4/5) ===
  LGB=0.74034  CAT=0.74028  XGB=0.74045  Rank=0.74065

=== Seed 777 (5/5) ===
  LGB=0.74045  CAT=0.74007  XGB=0.74050  Rank=0.74063

총 소요: 31.7분
평균 OOF  LGB=0.74073  CAT=0.74037  XGB=0.74078


## 4. 앙상블 & 결과 비교

In [12]:
oofs  = np.stack([oof_lgb,  oof_cat,  oof_xgb],  axis=1)
tests = np.stack([test_lgb, test_cat, test_xgb], axis=1)
aucs  = np.array([auc_lgb, auc_cat, auc_xgb])

results = {}

results['Simple Average'] = (
    roc_auc_score(y_train, oofs.mean(axis=1)),
    oofs.mean(axis=1), tests.mean(axis=1))

w_auc = aucs / aucs.sum()
results['AUC-weighted'] = (
    roc_auc_score(y_train, (oofs * w_auc).sum(axis=1)),
    (oofs * w_auc).sum(axis=1), (tests * w_auc).sum(axis=1))

oof_ranks  = np.stack([rankdata(oofs[:, i])  for i in range(3)], axis=1)
test_ranks = np.stack([rankdata(tests[:, i]) for i in range(3)], axis=1)
results['Rank Average'] = (
    roc_auc_score(y_train, oof_ranks.mean(axis=1)),
    oof_ranks.mean(axis=1), test_ranks.mean(axis=1))

def optuna_obj(trial):
    w = np.array([trial.suggest_float('w_lgb', 0, 1),
                  trial.suggest_float('w_cat', 0, 1),
                  trial.suggest_float('w_xgb', 0, 1)])
    w /= w.sum()
    return roc_auc_score(y_train, (oofs * w).sum(axis=1))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_obj, n_trials=300, show_progress_bar=False)
best_w = np.array([study.best_params['w_lgb'],
                   study.best_params['w_cat'],
                   study.best_params['w_xgb']])
best_w /= best_w.sum()
results['Optuna Weights'] = (
    roc_auc_score(y_train, (oofs * best_w).sum(axis=1)),
    (oofs * best_w).sum(axis=1), (tests * best_w).sum(axis=1))

BASELINE = 0.74068
print('=' * 65)
print(f'  평균 OOF: LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')
print(f'  EXP-020 기준선: {BASELINE}')
print('-' * 65)
best_method, best_auc, best_oof_pred, best_test = '', 0, None, None
for method, (auc, oof_pred, test_pred) in results.items():
    flag = ' ←' if auc > max(aucs) else ''
    print(f'  {method:20s}: {auc:.5f}  ({auc-BASELINE:+.5f} vs EXP-020){flag}')
    if auc > best_auc:
        best_auc, best_method, best_oof_pred, best_test = auc, method, oof_pred, test_pred
print('=' * 65)
print(f'\n최적: {best_method}  AUC={best_auc:.5f}')
print(f'Optuna 가중치  LGB={best_w[0]:.3f}  CAT={best_w[1]:.3f}  XGB={best_w[2]:.3f}')

  평균 OOF: LGB=0.74073  CAT=0.74037  XGB=0.74078
  EXP-020 기준선: 0.74068
-----------------------------------------------------------------
  Simple Average      : 0.74080  (+0.00012 vs EXP-020) ←
  AUC-weighted        : 0.74080  (+0.00012 vs EXP-020) ←
  Rank Average        : 0.74081  (+0.00013 vs EXP-020) ←
  Optuna Weights      : 0.74082  (+0.00014 vs EXP-020) ←

최적: Optuna Weights  AUC=0.74082
Optuna 가중치  LGB=0.347  CAT=0.154  XGB=0.499


## 5. Submission 저장 & 실험 기록

In [13]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = best_test
auc_str   = f'{best_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

저장: ../data/submissions/submission_exp028_조여진_07408.csv


In [14]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (best_oof_pred >= 0.5).astype(int)
params_str = (f'EXP-020 파라미터 | seeds={SEEDS} | best={best_method}')
NOTES    = (f'시드 앙상블: {len(SEEDS)}시드 × 3모델 = {len(SEEDS)*3}개 | '
            f'seeds={SEEDS}')
INSIGHTS = (f'EXP-020 대비 {best_auc-BASELINE:+.5f} | '
            f'LGB={auc_lgb:.5f} CAT={auc_cat:.5f} XGB={auc_xgb:.5f} | '
            f'Optuna: LGB={best_w[0]:.3f} CAT={best_w[1]:.3f} XGB={best_w[2]:.3f}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'SeedEnsemble(LGB+CAT+XGB, 5seeds)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'v1', X_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/24_seed_ensemble_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-028 기록 완료 (row 26)
